# SignLearn Data Exploration
This notebook explores the raw ASL Alphabet and Digits datasets to verify integrity, check class distributions, and identify potential issues.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import seaborn as sns

## 1. Directory Structure Check

In [ ]:
DATA_DIR = '../data/raw'
datasets = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]
print(f"Found datasets: {datasets}")

## 2. ASL Digits Exploration

In [ ]:
digits_path = os.path.join(DATA_DIR, 'digits')
if os.path.exists(digits_path):
    classes = sorted(os.listdir(digits_path))
    counts = {c: len(os.listdir(os.path.join(digits_path, c))) for c in classes if os.path.isdir(os.path.join(digits_path, c))}
    
    df_digits = pd.DataFrame(list(counts.items()), columns=['Class', 'Count'])
    plt.figure(figsize=(10, 5))
    sns.barplot(data=df_digits, x='Class', y='Count')
    plt.title('ASL Digits Class Distribution')
    plt.show()
else:
    print("Digits dataset not found.")

## 3. Sample Visualization

In [ ]:
def plot_samples(base_path, num_samples=5):
    classes = sorted([d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))])
    plt.figure(figsize=(15, 3))
    for i, cls in enumerate(classes[:num_samples]):
        img_name = os.listdir(os.path.join(base_path, cls))[0]
        img_path = os.path.join(base_path, cls, img_name)
        img = Image.open(img_path)
        plt.subplot(1, num_samples, i+1)
        plt.imshow(img)
        plt.title(cls)
        plt.axis('off')
    plt.show()

if os.path.exists(digits_path):
    plot_samples(digits_path)

## 4. Integrity Checks

In [ ]:
def check_integrity(path):
    corrupted = []
    sizes = []
    for root, dirs, files in os.walk(path):
        for f in files:
            if f.endswith(('.jpg', '.png', '.jpeg')):
                f_path = os.path.join(root, f)
                try:
                    with Image.open(f_path) as img:
                        img.verify()
                        sizes.append(img.size)
                except Exception:
                    corrupted.append(f_path)
    return corrupted, set(sizes)

if os.path.exists(digits_path):
    corrupted, unique_sizes = check_integrity(digits_path)
    print(f"Corrupted files: {len(corrupted)}")
    print(f"Unique image sizes: {unique_sizes}")